# Introduction to Transfer Learning


### **1. Overview and Learning Goals**

In this notebook, we take our first deep dive into **transfer learning** — a powerful idea that allows us to reuse pretrained deep networks instead of training from scratch.

Training deep convolutional networks from the ground up typically requires **millions of labeled images** and extensive compute resources. In contrast, pretrained models like *ResNet* have already learned general-purpose visual features (edges, textures, patterns) from massive datasets such as ImageNet. We can leverage these features for new tasks where data is limited — this is the essence of transfer learning.

In this notebook, we will:

- **Load a pretrained CNN** (ResNet18 trained on ImageNet).
- **Freeze** its feature extraction layers to preserve learned representations.
- **Replace** the final classifier head to match our 10-category Caltech subset.
- **Visualize and interpret** what pretrained models have already learned.
- **Run a zero-shot forward pass** to see how the model behaves “out-of-the-box.”

> The emphasis here is on **understanding concepts**, **examining features**, and **establishing a transfer learning pipeline**.  
> We will *not* train or fine-tune the network yet — that will come in NB02 and NB03.

Before we begin, let’s ensure reproducibility and consistent device setup.


In [1]:
# Reproducibility setup (same pattern used across projects)
import torch, random, numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[info] Using device: {device}")
print(f"[info] Random seed fixed at {SEED}")


[info] Using device: cpu
[info] Random seed fixed at 42


### **2. The Intuition Behind Transfer Learning**

When we train a deep convolutional neural network (CNN) from scratch, the model gradually learns to detect increasingly complex visual features:

- **Early layers** capture low-level patterns such as **edges**, **color gradients**, and **simple textures**.  
- **Middle layers** combine those primitives into **shapes** and **object parts**.  
- **Deeper layers** capture **high-level concepts** like faces, animals, or tools.

This hierarchical learning structure means that **the early layers are general-purpose**, useful across many visual tasks — while **the later layers become specialized** for the dataset they were trained on (e.g., ImageNet).

> Think of it like human learning: once we learn to recognize basic shapes (circles, rectangles), we can easily recognize new objects that combine them — learning to spot *trucks* becomes easier once we already know *cars*.

Because of this, we rarely need to train a CNN from scratch. Instead, we:
1. **Reuse (freeze)** the generic, pretrained convolutional layers that already extract useful visual features.
2. **Replace (train)** only the final classifier head for our specific task.

The benefits are substantial:
- **Faster convergence** — fewer parameters to train.
- **Lower data requirements** — we can work with small datasets.
- **Better generalization** — pretrained filters already encode rich, transferable visual knowledge.

However, we must be careful about **domain shift**.  
If our new dataset (e.g., plant leaves or medical scans) is very different from the original (e.g., natural photos), the pretrained features might not transfer perfectly — requiring partial fine-tuning or new adaptation strategies.

We will see these ideas in action by visualizing how pretrained networks “see” the world in upcoming sections.


### **3. Dataset Overview: Caltech-101 Ten-Class Teaching Subset**

Before diving into pretrained models, we first familiarize ourselves with the dataset we’ll be using.

We will work with a **10-class subset** of the **Caltech-101 dataset**, curated specifically for teaching and lightweight CPU experimentation.  
This subset contains:

- 10 diverse categories: **airplanes, chair, motorbike, watch, camera, butterfly, faces, lotus, panda, laptop**  
- Around **200–300 images per class**, capped for consistency  
- Images resized to **128 × 128** for efficient processing  
- Pre-split into **train / validation / test (70 % / 15 % / 15 %)**

This subset strikes a balance between **diversity** and **practicality** — it allows us to run deep learning experiments on ordinary hardware while retaining enough visual variety to explore meaningful transfer learning behavior.

> We will first inspect the manifest and verify that the splits and file structure are consistent, then visualize a small grid of images to get an intuitive feel for the data.


**Verify and Summarize the Dataset**

In [ ]:
import os, json, pandas as pd
from pathlib import Path

# Dataset paths (generated by the prep cell earlier)
DATA_ROOT = Path("./data/caltech101_10")
MANIFEST = DATA_ROOT / "manifest.csv"
CLASS_NAMES_JSON = DATA_ROOT / "class_names.json"

# Basic checks
assert DATA_ROOT.exists(), f"Dataset folder not found at {DATA_ROOT}"
assert MANIFEST.exists() and CLASS_NAMES_JSON.exists(), "Manifest or class_names.json missing"

# Load metadata
df = pd.read_csv(MANIFEST)
with open(CLASS_NAMES_JSON, "r", encoding="utf-8") as f:
    class_names = json.load(f)

print(f"[info] Found {len(df)} total images")
print(f"[info] Class names ({len(class_names)}): {class_names}")

# Summary by split
print("\n[summary] Images per split:")
print(df["split"].value_counts())

# Per-class per-split counts
counts = df.groupby(["split", "class_name"]).size().unstack(fill_value=0)
print("\n[summary] Per-class counts by split:")
display(counts)

# Check for any missing files
missing = [p for p in df["filepath"] if not Path(p).exists()]
print(f"\n[check] Missing files: {len(missing)} (should be 0)")


**Visualize Sample Images**

In [ ]:
import matplotlib.pyplot as plt
import math
from PIL import Image
from torchvision import transforms

# Basic image loader
tfm_preview = transforms.Compose([
    transforms.Resize(128),
    transforms.CenterCrop(128),
])

# Sample a few random training images
sample_df = df[df["split"]=="train"].sample(12, random_state=42).reset_index(drop=True)

cols, rows = 6, 2
plt.figure(figsize=(cols*2.2, rows*2.2))

for i, row in sample_df.iterrows():
    img = Image.open(row["filepath"]).convert("RGB")
    img = tfm_preview(img)
    plt.subplot(rows, cols, i+1)
    plt.imshow(img)
    plt.title(row["class_name"], fontsize=9)
    plt.axis("off")

plt.suptitle("Sample images from Caltech-101 (10-class subset)", fontsize=12)
plt.tight_layout()
plt.show()


By visually inspecting a few samples, we confirm that our curated Caltech-101 subset is diverse and cleanly structured.

Each class exhibits significant **intra-class variation** (different shapes, poses, lighting) — this diversity will be useful when testing how well pretrained features generalize to new categories.

In the next section, we will define consistent **transforms** and **data loaders** that standardize image size, apply normalization, and prepare batches for model input.


### **4. Preprocessing and Data Pipelines**

Before we can feed images into a pretrained model, we need to ensure they are transformed into the **format and scale expected by the network**.

Most pretrained CNNs (such as ResNet, VGG, or AlexNet) are trained on **ImageNet**, a large dataset of 1.2 million images. During that training, images were standardized in a specific way:
- resized and cropped to a fixed resolution,
- converted to tensors (values between 0 and 1),
- and **normalized** using the ImageNet dataset statistics.

Formally, each channel of the image tensor (R, G, B) is normalized as:
$$
x' = \frac{x - \mu}{\sigma}
$$
where $\mu$ and $\sigma$ are the channel-wise mean and standard deviation from ImageNet:

$$
\mu = [0.485, 0.456, 0.406], \quad \sigma = [0.229, 0.224, 0.225]
$$

This ensures that inputs to the pretrained model fall within the same scale as those used during pretraining — otherwise, the learned filters would respond incorrectly.

In our case, we will:
1. Resize all images to **128 × 128** (for computational efficiency),
2. Apply `CenterCrop(128)`,
3. Convert to tensors,
4. Normalize using **ImageNet mean and std**.

We will also define **DataLoaders** for the train, validation, and test splits.  
These loaders batch our data, shuffle it (for training), and prepare tensors for the GPU or CPU.  
Finally, we will visualize a mini-batch to confirm that everything works as expected.


**Define Transforms and Constants**

In [ ]:
from torchvision import transforms

# ImageNet normalization constants
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Define the preprocessing pipeline
transform = transforms.Compose([
    transforms.Resize(128),
    transforms.CenterCrop(128),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print("[info] Transformation pipeline ready.")

**Build Dataset Objects and DataLoaders**

In [ ]:
from torchvision import datasets
from torch.utils.data import DataLoader

# Load dataset from prepared folders
train_dir = DATA_ROOT / "train"
val_dir   = DATA_ROOT / "val"
test_dir  = DATA_ROOT / "test"

train_ds = datasets.ImageFolder(train_dir, transform=transform)
val_ds   = datasets.ImageFolder(val_dir, transform=transform)
test_ds  = datasets.ImageFolder(test_dir, transform=transform)

# DataLoaders (small batch size for CPU runs)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)

print(f"[info] Dataset sizes → train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")
print(f"[info] Classes: {train_ds.classes}")


In [ ]:
**Visualize a Mini-Batch**

In [ ]:
import matplotlib.pyplot as plt
import torchvision

# Helper function to unnormalize images for display
def unnormalize(img_tensor):
    img = img_tensor.clone()
    for c, (m, s) in enumerate(zip(IMAGENET_MEAN, IMAGENET_STD)):
        img[c] = img[c] * s + m
    return img.clamp(0, 1)

# Fetch one mini-batch
images, labels = next(iter(train_loader))

# Unnormalize for display
images_disp = [unnormalize(img) for img in images[:8]]

# Create grid
grid = torchvision.utils.make_grid(images_disp, nrow=8)
plt.figure(figsize=(14, 2.5))
plt.imshow(grid.permute(1, 2, 0))
plt.title("Mini-batch samples after preprocessing")
plt.axis("off")
plt.show()

print(f"Labels for these samples: {[train_ds.classes[i] for i in labels[:8].tolist()]}")


Our preprocessing pipeline now mirrors the ImageNet setup expected by pretrained models.  

This ensures that the pixel intensities and distributions are consistent with what the pretrained filters “expect to see.”  

By inspecting the mini-batch visualization, we verify that images are correctly sized, centered, and normalized.  

We are now ready to load a pretrained model (ResNet18) and adapt it for our Caltech-101 subset.

### **5. Loading and Inspecting a Pretrained CNN**

We now load a **pretrained convolutional neural network** (CNN) to understand what transfer learning builds upon.  
In this notebook, we will use **ResNet-18**, a smaller but powerful variant of the *Residual Network* family trained on the **ImageNet** dataset.

ResNet models introduced **skip (residual) connections** that allow gradients to flow more easily through very deep architectures.  
Instead of learning a direct mapping $H(x)$, each residual block learns a *residual function* $F(x) = H(x) - x$, which simplifies optimization.  
The block then outputs:

$$
y = F(x) + x
$$

This additive shortcut helps avoid the **vanishing-gradient problem** and enables much deeper networks to train successfully.

The pretrained **ResNet-18** was trained to classify **1000 ImageNet categories** (from dogs and cats to tools and furniture).  
Its structure consists of:
- An initial **convolutional stem** (Conv–BN–ReLU–Pooling),
- Four **residual stages** that progressively reduce spatial dimensions and increase feature depth,
- A **global average pooling** layer that flattens the spatial map into a feature vector,
- And a final **fully-connected (FC)** layer mapping to 1000 output classes.

By loading the pretrained weights, we inherit knowledge about **edges, textures, and shapes** that were already learned from millions of images.

We will now:
1. Load `torchvision.models.resnet18(weights='IMAGENET1K_V1')`
2. Print a summary of the model’s architecture and parameters.
3. Build intuition for how the depth of the network relates to the level of feature abstraction.


**Load the Pretrained Model**

In [3]:
import torch
import torchvision
from torchvision import models

# Load pretrained ResNet18 model
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

print("[info] Pretrained ResNet-18 loaded successfully.")

ModuleNotFoundError: No module named 'torchvision'

**Inspect the Architecture**

In [ ]:
from torchsummary import summary

# Move to CPU (since we are in concept notebook)
device = torch.device("cpu")
model.to(device)

# Display model structure and parameter count
summary(model, input_size=(3, 128, 128))

layer_hierarchy = [name for name, _ in model.named_children()]
print("Top-level children:", layer_hierarchy)

In [ ]:
**Visualize the Layer Hierarchy**

In [ ]:
# Show hierarchical layer structure
print(model)

# Quick count of total and trainable parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n[summary] Total parameters: {total_params:,}")
print(f"[summary] Trainable parameters: {trainable_params:,}")

By inspecting the printed architecture, we can trace how **ResNet-18** transforms an input image through successive convolutional and residual blocks.

- Early layers detect **edges** and **colors**.  
- Mid-level layers combine them into **shapes** and **object parts**.  
- Deep layers encode **abstract semantic features** such as “animal face” or “wheel.”

The skip connections ($y = F(x) + x$) make training such deep networks feasible, preserving gradient flow even through dozens of layers.  

This pretrained network now acts as a powerful **feature extractor** — in the next section, we will selectively **freeze its layers** and **replace the final classifier** to adapt it for our 10-class Caltech-101 subset.

### **6. Freezing Layers and Swapping the Classifier Head**

Our goal is to **reuse** the pretrained feature extractor and only learn a **small new classifier** for our 10-class Caltech subset. Practically, we:

1. **Freeze** all pretrained parameters by setting `requires_grad = False`. This preserves the feature extractor as-is and reduces the number of trainable parameters dramatically.
2. **Replace** the final fully-connected (FC) layer with a new head sized to our task: `nn.Linear(in_features, 10)`.
3. **Re-initialize** the new head (e.g., Kaiming initialization for ReLU-based networks) so that it starts from a reasonable random state.

Why freeze first? With a small dataset, training the entire network risks **overfitting** and **catastrophic forgetting**. By freezing the backbone and training only the head, we perform **feature extraction**: we map the rich, pretrained features to our class labels efficiently and reliably. In later notebooks, we will **unfreeze upper blocks** and fine-tune carefully with a smaller learning rate to adapt features while protecting what the model already knows.
